## Kalman Filter

This section reviews the design and implementation of the Kalman filter. Its purpose is to estimate the robot’s position at each step by combining information from odometry and the camera.

The Kalman filter first performs a prediction step, where it estimates the robot’s next position based on its current pose and motion commands. However, odometry is not perfectly accurate, and small errors quickly accumulate, causing the Thymio to drift from its true position.

The update (correction) step compensates for this drift. When a camera measurement is available, the filter incorporates it to correct the predicted pose, pulling the estimate back toward the robot’s real location. This continuous predict–correct cycle keeps the robot accurately localized as it moves toward its goal.

### 1. State Definition

The goal of the EKF is to estimate the 2D pose of the Thymio robot.  
We define the state vector as:

$$
\begin{bmatrix}
x \\
y \\
\theta
\end{bmatrix}
$$

With:

- $x$: robot position along the horizontal axis, in [mm] 
- $y$: robot position along the vertical axis, in [mm]
- $\theta$: robot orientation, wrapped to $[- \pi, \pi]$, in [rad]

And the inputs to the motion models being:

- $v$: the linear speed
- $w$: the angular speed

Since the Thymio does not directly measure these two quantities, they were computed in the **Motion Control** module and passed to the EKF as inputs.

### 2. Motion Prediction Model

The model is based on an Extended Kalman Filter (EKF) since our system is nonlinear.
The prediction step uses a unicycle/differential-drive kinematic model, which matches the physical structure and motion constraints of the Thymio robot.

The Thymio has exactly two actuated wheels, making it a differential-drive platform. Its motion; moving forward/backward, rotating on the spot, and following curved trajectories; is accurately described by the unicycle model. Other models, such as the bicycle model or full dynamic models, allow lateral motion or include forces the Thymio does not experience, and are therefore not appropriate here.

We follow the standard kinematic equations:

$$
\begin{aligned}
x_{k+1} &= x_k + T_s\, v \cos(\theta_k), \\[4pt]
y_{k+1} &= y_k + T_s\, v \sin(\theta_k), \\[4pt]
\theta_{k+1} &= \theta_k + T_s\, \omega.
\end{aligned}
$$


The presence of the nonlinear terms $\cos(\theta_k)$ and $\sin(\theta_k)$ makes the system nonlinear, which is why an EKF is required instead of a standard Kalman Filter.

Next, the Jacobian of the motion model with respect to the state is:

$$
A =
\begin{bmatrix}
1 & 0 & -T_s\, v \sin\theta \\
0 & 1 & \;\;T_s\, v \cos\theta \\
0 & 0 & 1
\end{bmatrix}
$$

Finally, the covariance prediction step is given by:

$$
P_{k+1|k} = A\, P_{k|k}\, A^\top + Q
$$

where $Q$ represents the process noise, modelling uncertainty in the motion prediction. \
The units are $[mm^2]$ for $\sigma_x^2$ and $\sigma_y^2$ and $[rad^2]$ for $\sigma_\theta^2$.

The complete implementation of this prediction step is provided in the following code section:

```python

def ekf_predict(x, P, v, omega, Ts, Q):
    
    ...
    
    # Nonlinear motion model (discrete-time)
    x_pos_pred = x_pos + Ts * v * np.cos(theta)          
    y_pos_pred = y_pos + Ts * v * np.sin(theta)          
    theta_pred = wrap_angle(theta + Ts * omega)          
                
    ... 

    # Jacobian of the motion model w.r.t. state (A matrix)
    A = np.array([
        [1, 0, -Ts * v * np.sin(theta)],                 
        [0, 1,  Ts * v * np.cos(theta)],                 
        [0, 0,  1]                                       
    ])

    # Covariance prediction using linearized model
    P_pred = A @ P @ A.T + Q                             

    return x_pred, P_pred      

```

This function takes as inputs the state vector $x = [x,\,y,\,\theta]^T$, the current covariance matrix $P$,
the linear and angular velocities $v$ and $\omega$, the sampling time $T_s$, and the process noise matrix $Q$.
It returns the predicted state $x_{\text{pred}}$ and the predicted covariance $P_{\text{pred}}$.


#### Covariance
The covariance matrix is a measure of the uncertainty and correlation between the state variables.
The diagonal terms represent the uncertainty of each state component, while the off-diagonal terms indicate
how errors in one variable influence the others. For example, during a turning motion, the uncertainty in
$\theta$ affects both $x$ and $y$, increasing their correlation.

If the covariance is large, the Kalman filter interprets the prediction as unreliable and therefore gives
more weight to the measurement. Conversely, if the covariance is small, the filter trusts the prediction more.

Source: [The Kalman Filter (18 of 55) What is a Covariance Matrix?](https://youtu.be/mYAsKbwqGv0?si=bKLiNsdeX1D0KAqB)

#### Angle Wrap
The prediction step also uses a helper function to wrap angles:

```python
def wrap_angle(a):
    """
    Wraps an angle to the range [-pi, pi].

    """
    return (a + np.pi) % (2 * np.pi) - np.pi
```
This ensures that all angle computations remain within the interval $[- \pi, \pi]$,
avoiding discontinuities that would otherwise destabilize the filter.

### 3. Measurement (Update) Model

When the camera is active it has to update the prediction made in the step before that, using the camera measurement.

At each timestep, the camera provides a direct measurement of the robot pose:

$$
z =
\begin{bmatrix}
x \\
y \\
\theta
\end{bmatrix}
$$

where:
-  $x$ and $y$ are in millimeters (mm),
- $\theta$ is in radians (rad).

Since the camera directly observes all three state variables, the measurement model is simply:
$$
H = I_3,
$$
meaning the measurement is a direct reading of the state without transformation.

#### Innovation
The innovation (also called residual) is the difference between the measurement and the predicted state:
$$
y = z - H\, x_{k|k-1}.
$$
It expresses how far the predicted pose is from the detected pose, and in the same way, how wrong was the prediction compared to the measurement.

**What is a good Innovation?** A good innovation is not necessarily small but statistically consistent.
For a well-tuned EKF, the innovations oscillate around zero, remain within the expected uncertainty bounds (typically within $\pm 2\sqrt{S}$), and show no long-term drift.  
Occasional peaks are normal and usually occur during sharp turns, rapid motion changes, or brief camera interruptions.

#### Innovation Covariance
The uncertainty associated with the innovation is:
$$
S = H\, P_{k|k-1}\, H^\top + R,
$$
where:
- $P_{k|k-1}$ is the predicted covariance $[\text{mm}^2, \text{rad}^2]$,
- $R$ is the camera measurement noise covariance,
- $S$ has the same units as the measurement:  
$\text{mm}^2$ for $x,y$ and $\text{rad}^2$ for $\theta$.

The innovation covariance $S$ plays a similar role to the state covariance $P$. 
It quantifies how uncertain we are about the innovation itself.  
A small value of $S$ means the innovation is reliable and should be trusted, whereas a large value indicates high uncertainty, and the filter should rely less on the innovation.


#### Kalman Gain
The Kalman gain determines how much the filter should correct the prediction using the measurement:
$$
K = P_{k|k-1}\, S^{-1}.
$$
A large $K$ means the filter trusts the camera more than the prediction while
a small $K$ means the opposite.

#### Updated State Estimate 
The corrected state estimate is computed as:
$$
x_{k|k} = x_{k|k-1} + K\,y.
$$

#### Updated Covariance (Joseph Form)
To maintain numerical stability, the Joseph form is used:
$$
P_{k|k}
=
(I - KH)\, P_{k|k-1}\, (I - KH)^\top
+
K\,R\,K^\top.
$$

The diagonal values of $P_{k|k}$ represent the updated uncertainty:

- $\sigma_x^2 $ and $\sigma_y^2 $ in $\text{mm}^2$,
- $\sigma_\theta^2 $ in $\text{rad}^2$.

Once again, since the orientation must remain within a valid range, the angle is wrapped after every update:
$$
\theta \in [-\pi, \pi].
$$
This prevents discontinuities when crossing the $\pm\pi$ boundary and ensures consistent filter behavior.

We can observe that each step of the update stage builds directly on the previous one, ensuring that the final corrected state is as accurate as possible. 
The process ends with the covariance update, which adjusts the filter's confidence: 
a smaller covariance means we become more certain about the corrected state thanks to the measurement.

Source: [Wikipedia - Kalman filter](https://en.wikipedia.org/wiki/Kalman_filter)


The implementation of the update step is given below:

```python
def ekf_update_camera(x_pred, P_pred, z_cam, R_cam):

    H = np.eye(3)                                       

    # Innovation (measurement residual) based on predicted state
    y = z_cam - H @ x_pred                              
    y[2, 0] = wrap_angle(y[2, 0])                       

    # Innovation covariance
    S = H @ P_pred @ H.T + R_cam                        

    # Kalman gain
    K = P_pred @ H.T @ np.linalg.inv(S)                 

    # Updated state estimate
    x_upd = x_pred + K @ y                              
    x_upd[2, 0] = wrap_angle(x_upd[2, 0])               

    # Updated covariance matrix (Joseph form for better numerical stability)
    I = np.eye(3)                                       
    P_upd = (I - K @ H) @ P_pred @ (I - K @ H).T + K @ R_cam @ K.T  

    return x_upd, P_upd, y

```
This function takes as inputs:
- $x_{\text{pred}}$: the predicted state vector $[x,\,y,\,\theta]^T$,
- $P_{\text{pred}}$: the predicted covariance matrix,
- $z_{\text{cam}}$: the camera measurement, already expressed in millimeters and radians,
- $R_{\text{cam}}$: the measurement noise covariance matrix.

It returns:

- $P_{\text{upd}}$: the updated covariance representing the new uncertainty,
- $y$: the innovation, i.e., the difference between prediction and measurement.
- $x_{\text{upd}}$: the updated (corrected) state estimate,

This completes the camera correction step. 
In the next section, we combine both the prediction and the update into a single function, forming the full EKF loop that is executed at every timestep.

### 4. EKF Step

Now that both stages of the EKF have been presented individually, we can combine them into the complete filtering logic. At every iteration, the filter first performs the prediction step (using odometry), and then the update step 
(whenever a camera measurement is available). This decision is implemented bychecking that the camera is active (i.e.~the Thymio is detected) and that a valid measurement has been received.

This function is executed at every sampling instant and ensures that the EKF continuously maintains the best estimate of the Thymio's pose.

```python
def ekf_step(x_prev, P_prev, v_wheel, omega_wheel, z_cam, camera_on, Ts, Q, R_cam):
```

The function begins by calling the prediction model to compute the predicted state $x_{k|k-1}$ and the predicted covariance $P_{k|k-1}$. If a camera measurement is available, the update step is then applied through the 
$ekf\_update\_camera$ function. Otherwise, the filter simply carries the prediction forward for that timestep.

The most important parts of the implementation are shown below:

```python

def ekf_step(x_prev, P_prev, v_wheel, omega_wheel, z_cam, camera_on, Ts, Q, R_cam):

...

# ---- PREDICTION ----
    x_pred, P_pred = ekf_predict(x_prev, P_prev, v_wheel, omega_wheel, Ts, Q)  
    
...

# ---- UPDATE (if camera is on and measurement is available) ----
    if camera_on and (z_cam is not None):         

        ...

        # EKF camera update
        x_new, P_new, innovation = ekf_update_camera(x_pred, P_pred, z, R_cam)  

        ...

    else:
        # No measurement update: just carry prediction 
        x_new, P_new = x_pred, P_pred                  
        ...                          

return x_new, P_new   
```

All relevant quantities (state estimate, covariance, innovation, and the indicator of whether a measurement was used) are logged at each iteration. These logs are later used to generate the plots shown in the analysis section: 
covariance evolution, innovation history, and error versus confidence bounds. They were essential for tuning the filter (particularly the matrices $Q$ and $R$) and for identifying potential issues in the robot's behavior.

### 5. Plotting Section

This part of the implementation allowed us to generate informative plots that helped us understand the Thymio’s behavior and evaluate whether the filter parameters were properly tuned. These visualizations also revealed additional effects and behaviors of the robot that became apparent through the plotted data.

#### Covariance Evolution

This plot shows the evolution of the three diagonal elements of the covariance 
matrix $P_{k|k}$: the uncertainty in $x$, $y$, and $\theta$. 
It allows us to visualize how confident the EKF becomes over time.

##### Global + Local

In both the normal (global navigation) and local avoidance cases, the covariance evolution remained stable and within low values. This is exactly what we aim for, as it indicates that the model behaves consistently and the EKF maintains good confidence in its estimates.

We did notice small, recurrent spikes that appear to come from occasional delays or missed updates between the camera and odometry. Each time a camera frame is skipped, the covariance grows slightly, then drops back down as soon as the next measurement arrives. These spikes are extremely small and have no practical impact on the robot’s motion or decisions; they are simply worth mentioning for completeness.

In contrast, as we will see in the “blind” experiment (when the camera is turned off), these small “uncertainty spikes” are negligible compared to the rapid and unbounded covariance growth that occurs when the robot must rely solely on odometry.


<p align="center">
  <img src="images/normalcov.png" width="100%">
</p>

##### Blind 

In the blind experiment, we observe that around $t\approx 12\,\text{s}$ the robot loses access to the camera and must rely solely on odometry. From this moment on, the filter becomes increasingly uncertain about the robot's true position.At the end of the blind period, the covariance in the $y$--direction reaches approximately
$$
\sigma_y^2 \approx 15000,
\qquad
\Rightarrow \qquad
\sigma_y \approx \sqrt{15000} \approx 12.2\,\text{cm}.
$$

At first sight this may appear very large, but the value is consistent with the conditions of the experiment.  
The robot travels at roughly
$$
v \approx 6.2\,\text{cm/s},
$$
and during the blind interval (about $12$--$13$ seconds), even a small angular uncertainty can accumulate into a large lateral drift.  
The orientation covariance reaches
$$
\sigma_\theta^2 \approx 0.08 
\qquad\Rightarrow\qquad
\sigma_\theta \approx \sqrt{0.08}
= 0.283\,\text{rad}
\approx 16.2^\circ.
$$

With this level of angular uncertainty, the robot could easily drift several centimetres sideways while moving forward, which explains the observed $12\,\text{cm}$ positional uncertainty.  
Thus, although the covariance grows rapidly, its magnitude is realistic given that the robot is navigating blindly using only odometry.

<p align="center">
  <img src="images/blindcov.png" width="100%">
</p>

We also observe that the small spikes seen earlier in the normal and local navigation experiments do not appear here, as they are several orders of magnitude smaller than the uncertainty accumulated during blind operation. They can therefore be safely neglected.

#### Innovation History

As said earlier, the innovation represents the difference  between the measurement and the prediction. The innovation history plot shows how this residual evolves over time for all three state variables.

A well-behaved innovation should oscillate around zero and remain within the bounds defined by the expected noise (typically within $\pm 2\sqrt{S}$). Large spikes indicate moments when the camera update corrects a significant prediction error, often caused by turning, temporary occlusions, or short  periods without camera visibility.

##### Global 

In the normal case, the innovation history behaves exactly as expected: the residuals oscillate around zero with no visible long-term drift or trend. This indicates that the EKF predictions and camera measurements remain consistent with each other and that the filter is properly tuned. 

However, it is sometimes possible to observe a small bias in the innovation, typically on the $x$ component in our case. This can arise from a mismatch between the camera and odometry mappings (e.g.\ imperfect homography or scale), or from a slight geometric offset of the ArUco marker with respect to the Thymio's wheel axis, which appears as a constant shift in $x$. Nevertheless, if we mentally remove this bias, the innovation signals are still centered around zero, which indicates that the filter remains statistically consistent.

<p align="center">
  <img src="images/normalinno.png" width="100%">
</p>

##### Blind

In the blind case, we observe that the innovation remains at zero during the entire period where the robot is not seen by the camera. This is completely expected, since no measurement is available and the EKF therefore performs only the prediction step.

Once the robot becomes visible again, the innovation values suddenly become much larger than in normal operation. This indicates that the difference between the Thymio's predicted pose and the camera measurement was significant after the
blind interval. The effect is particularly noticeable on the $y$ component, showing that most of the accumulated odometry drift occurred along this axis.

<p align="center">
  <img src="images/blindinno.png" width="100%">
</p>

##### Error vs. Confidence Bounds

This plot compares the EKF estimation error (computed using camera-measured ground truth when available) with the predicted uncertainty represented by  $\pm 2\sigma$. For each state component, we plot:

$$
\text{error}(x), \quad \pm 2\sqrt{P_{xx}}
$$

and similarly for $y$ and $\theta$.

If the EKF is well tuned, the error curve should remain mostly inside the $\pm 2\sigma$ envelope. This indicates that the filter is statistically consistent: the prediction uncertainty matches the actual behavior of the robot.

We used this plot as the final validation step for the chosen parameters $Q$ and $R$. If the error frequently exceeded the theoretical bounds, we increased the corresponding variance in $Q$ or $R$ until consistency was achieved.

Among the three state components, the orientation $\theta$ is one of the most critical to evaluate. Even a small angular error can generate a large positional deviation over time, which makes $\theta$ the most important quantity to keep accurately estimated.

In the global (normal) navigation case, the EKF behaves exactly as expected. The plot shows that the angular error remains centered around zero and stays comfortably within the $\pm 2\sigma_{\theta}$ confidence bounds at all times.
This indicates that the prediction and the camera measurement are in good statistical agreement, with no sign of divergence or long–term drift. The confidence bounds remain stable, and the error oscillates naturally inside
them.

<p align="center">
  <img src="images/normalerrtetha.png" width="100%">
</p>

This behavior confirms that the Kalman filter tuning (the choice of $Q$, $R$ and $P$) was done correctly: the filter is neither overconfident nor underconfident, and the uncertainty predicted by the model matches the actual
estimation error observed during operation. 

### 7. Conclusion

To conclude, we have been able to implement a Kalman Filter that:

- successfully fused odometry and camera measurements, producing a stable and reliable pose estimate;
- behaved consistently during prediction: uncertainty increased when the robot was blind and decreased immediately once measurements were available;
- maintained innovations centered around zero, confirming statistical consistency and correct model behaviour;
- kept the estimation error within the $\pm 2\sigma$ bounds, demonstrating that the noise parameters $Q$, $R$, and $P$ were properly tuned;
- remained stable across all scenarios (normal motion, local avoidance, and blind runs) and rapidly corrected itself once the camera re-detected the robot.

Overall, the EKF greatly improved the robot’s navigation accuracy by correcting odometry drift and ensuring smooth, robust tracking of the Thymio’s pose.

